# PMXT EMA optimizer research notebook

This notebook runs a compact optimizer sweep derived from the public EMA optimizer, then replays the first holdout window with the selected parameters and emits HTML into `output/`.

Launch it from `make backtest` or `uv run python main.py` so the repo menu treats it like any other flat runner entrypoint.


In [ ]:
from dataclasses import replace
from pprint import pprint

from backtests._shared._optimizer import run_parameter_optimization
from backtests.polymarket_quote_tick_pmxt_ema_optimizer import NAME, OPTIMIZATION

NOTEBOOK_OPTIMIZATION = replace(
    OPTIMIZATION,
    name=f"{NAME}_research",
    max_trials=4,
    holdout_top_k=2,
)

summary = run_parameter_optimization(NOTEBOOK_OPTIMIZATION)
best_params = dict(summary.selected_params)
print("Selected params:")
pprint(best_params)
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

In [ ]:
from decimal import Decimal

from backtests._shared._experiments import build_replay_experiment, run_experiment
from backtests._shared._replay_specs import PolymarketPMXTQuoteReplay
from backtests.polymarket_quote_tick_pmxt_ema_crossover import (
    DETAIL_PLOT_PANELS,
    REPORT,
)
from backtests.polymarket_quote_tick_pmxt_ema_optimizer import (
    BASE_REPLAY,
    CHART_OUTPUT_PATH,
    DATA,
    EXECUTION,
    HOLDOUT_WINDOWS,
    NAME,
)

holdout_window = HOLDOUT_WINDOWS[0]
holdout_replay = PolymarketPMXTQuoteReplay(
    market_slug=BASE_REPLAY.market_slug,
    token_index=BASE_REPLAY.token_index,
    start_time=holdout_window.start_time,
    end_time=holdout_window.end_time,
)

holdout_experiment = build_replay_experiment(
    name=f"{NAME}_research_holdout",
    description="Notebook research flow using best optimizer params on the first holdout window",
    data=DATA,
    replays=(holdout_replay,),
    strategy_configs=[
        {
            "strategy_path": "strategies:QuoteTickEMACrossoverStrategy",
            "config_path": "strategies:QuoteTickEMACrossoverConfig",
            "config": {
                "trade_size": Decimal("5"),
                **best_params,
            },
        }
    ],
    initial_cash=NOTEBOOK_OPTIMIZATION.initial_cash,
    probability_window=NOTEBOOK_OPTIMIZATION.probability_window,
    min_quotes=NOTEBOOK_OPTIMIZATION.min_quotes,
    min_price_range=NOTEBOOK_OPTIMIZATION.min_price_range,
    execution=EXECUTION,
    report=REPORT,
    empty_message="No PMXT EMA crossover sims met the quote-tick requirements.",
    emit_html=True,
    chart_output_path=CHART_OUTPUT_PATH,
    detail_plot_panels=DETAIL_PLOT_PANELS,
)

results = run_experiment(holdout_experiment)
print(f"Generated {len(results)} holdout replay result(s).")
print("HTML output root:", CHART_OUTPUT_PATH)

<!-- prediction-market-backtesting:auto-embedded-html -->
Generated HTML artifacts will be embedded here after execution.
